# Notebook 06 — Comparativo Final: Value-Based vs Policy-Based

**Disciplina:** Modelos de Aprendizagem por Reforço  
**Aula:** 03 — Policy-Based Methods  
**Ambiente:** CartPole-v1 (Gymnasium)  
**Bibliotecas:** numpy, matplotlib, gymnasium

| | |
|---|---|
| **Aula** | Aula 03 — Métodos Baseados em Políticas |
| **Notebook** | 06 — Comparativo Final |
| **Seções** | 3.9 |
| **Tempo de leitura** | ~10 min |
| **Tempo de execução** | ~2 min |

**Pré-requisitos:** Notebooks 01–05 desta aula.

**Competências para o Desafio Final:** Selecionar o algoritmo adequado dado o tipo de ação, restrições de amostra e requisitos de estabilidade; reconhecer onde as famílias value-based e policy-based se complementam.

---

### Recapitulando

A Aula 03 percorreu a progressão completa dos métodos baseados em políticas: da **intuição de política parametrizada** (NB01) ao **gradiente de política com REINFORCE** (NB02), passando por **baseline e A2C** (NB03), **PPO com clipping** (NB04) até **DDPG e SAC para ações contínuas** (NB05). Este notebook consolida o que foi aprendido e traça a ponte para a Aula 04.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import rl_utils
rl_utils.info_versoes()
rl_utils.configurar_matplotlib()
rl_utils.definir_seeds(42)

Biblioteca           Versão
--------------------------------
gymnasium            1.0.0


torch                2.11.0+cpu
numpy                2.4.2
matplotlib           3.10.8
pandas               3.0.1


scikit-learn         1.8.0


## Bloco 1 — Contexto e pergunta central

As Aulas 02 e 03 percorreram duas grandes famílias de algoritmos de RL:

- **Value-based (Aula 02):** aprender $Q(s, a)$ e derivar a política a partir dos valores — de Programação Dinâmica ao DQN.
- **Policy-based (Aula 03):** aprender a política $\pi(a|s; \theta)$ diretamente — de REINFORCE ao SAC.

> **"Quando usar cada família? O que cada abordagem ganha e perde?"**

Este notebook consolida a comparação entre as duas famílias, mostra onde elas se complementam e aponta para os temas da Aula 04.

## Bloco 2 — Mini teoria

### Tabela síntese

| Critério | Value-Based | Policy-Based |
|---|---|---|
| Objeto de aprendizado | Função de valor $Q(s,a)$ ou $V(s)$ | Política $\pi(a|s; \theta)$ |
| Tipo de ação | Discreto (finito) | Discreto **ou** contínuo |
| Exploração | $\epsilon$-greedy, UCB | Estocástica (entropia, ruído) |
| Estabilidade | DQN: instável sem truques | PPO/SAC: mais estável |
| Eficiência amostral | Alta (*off-policy*, reutiliza experiências passadas, com replay) | Baixa (*on-policy*, aprende sobre a política executada) / Alta (*off-policy*: DDPG, SAC) |
| Convergência | Garantias tabulares; aprox. sem garantia | Convergência local (gradiente) |
| Quando preferir | Ações discretas, ambientes de jogos | Ações contínuas, robótica, controle |

### Progressão dos algoritmos — o que cada um resolve

Cada algoritmo desta aula surgiu para corrigir uma limitação do anterior. A tabela abaixo torna essa cadeia explícita:

| # | Algoritmo | Problema que resolve | Limitação que deixa em aberto |
|---|---|---|---|
| 1 | **REINFORCE** | Elimina o `argmax` sobre ações → suporta espaços discretos grandes e contínuos | Alta variância dos gradientes: cada episódio produz um sinal de atualização muito ruidoso |
| 2 | **A2C** (Actor-Critic) | Reduz a variância com baseline e função de vantagem $A(s,a) = G_t - V(s)$ | Atualizações ainda podem ser grandes demais e colapsar a política aprendida |
| 3 | **PPO** | Controla o tamanho das atualizações com clipping: $\min(r_t A_t, \text{clip}(r_t, 1{-}\varepsilon, 1{+}\varepsilon) A_t)$ | Exploração depende de ruído externo ou da entropia da distribuição — não é parte do objetivo |
| 4 | **DDPG** | Política determinística *off-policy* para ações contínuas; replay buffer aumenta a eficiência amostral | Sensível ao nível de ruído de exploração $\sigma$; instável sem ajuste cuidadoso de hiperparâmetros |
| 5 | **SAC** | Incorpora a entropia no objetivo: $J(\pi) = \mathbb{E}[r_t + \alpha \mathcal{H}(\pi)]$ → exploração automática e estável | Mais complexo: duas redes $Q$ + ajuste automático do coeficiente de temperatura $\alpha$ |

### Onde as famílias se encontram

Os melhores algoritmos modernos **combinam as duas abordagens**:
- **Actor-Critic** (A2C, PPO): o crítico (value-based) orienta o ator (policy-based).
- **DDPG / SAC / TD3**: política determinística/estocástica com crítico Q off-policy.
- **AlphaZero / MuZero**: política + função de valor + modelo do ambiente.

Essa convergência é o ponto de partida da Aula 04.

In [2]:
# %pip install numpy matplotlib gymnasium

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print(f"gymnasium: {gym.__version__}")
print("Ambiente pronto.")

gymnasium: 1.0.0
Ambiente pronto.


In [4]:
# ── Comparação visual: políticas baseline no CartPole ────────────
env = gym.make("CartPole-v1")

def avaliar_politica(politica_fn, n_episodios=50, seed_base=SEED):
    retornos = []
    for ep in range(n_episodios):
        obs, _ = env.reset(seed=seed_base + ep)
        ret = 0; terminado = False
        while not terminado:
            acao = politica_fn(obs)
            obs, r, term, trunc, _ = env.step(acao)
            terminado = term or trunc; ret += r
        retornos.append(ret)
    return retornos

def politica_aleatoria(obs):
    return env.action_space.sample()

def politica_heuristica(obs):
    angulo = obs[2]
    return 1 if angulo > 0 else 0

retornos_aleatorio  = avaliar_politica(politica_aleatoria)
retornos_heuristica = avaliar_politica(politica_heuristica)
env.close()

print(f"Política aleatória  — média: {np.mean(retornos_aleatorio):.1f}  +/- {np.std(retornos_aleatorio):.1f}")
print(f"Política heurística — média: {np.mean(retornos_heuristica):.1f}  +/- {np.std(retornos_heuristica):.1f}")

Política aleatória  — média: 18.9  +/- 8.4
Política heurística — média: 42.7  +/- 7.8


### Referência baseline — os números do chão

Esses dois valores definem o **piso de desempenho** para o CartPole-v1:

- **Aleatória (~19 ±8):** sem qualquer informação do estado — sorteio puro. Todo algoritmo treinado deve superar isso amplamente.
- **Heurística (~43 ±8):** regra manual "polo à direita → empurra à direita" — funciona porque o CartPole tem solução analítica simples, mas não generaliza para outros problemas.

O limiar de solução do CartPole-v1 é **475/500**. Os algoritmos da Aula 03 (REINFORCE, A2C, PPO) atingem esse limiar após treinamento — partindo do mesmo ponto inicial (~19) que a política aleatória. A visualização a seguir posiciona esses baseline no contexto da progressão completa.

In [ ]:
# ── Visualização comparativa ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Value-Based vs Policy-Based — comparativo", fontsize=14)

# --- Painel esquerdo: boxplot (caixa = Q1–Q3; linhas = min/max excluindo outliers) ---
ax = axes[0]
dados = [retornos_aleatorio, retornos_heuristica]
nomes = ["Aleatoria", "Heuristica (angulo)"]
bp = ax.boxplot(dados, tick_labels=nomes, patch_artist=True, widths=0.4)
# azul = referência sem estrutura; laranja = heurística manual
cores_box = ["#AEC6E8", "#FFBB78"]
for patch, cor in zip(bp['boxes'], cores_box):
    patch.set_facecolor(cor)
ax.axhline(y=475, color="green", linestyle="--", alpha=0.7, label="Limiar de solucao (475)")
ax.set_ylabel("Retorno acumulado")
ax.set_title("Desempenho no CartPole-v1 (50 eps, sem treinamento)")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")

# --- Painel direito: tabela visual de propriedades ---
ax2 = axes[1]
ax2.axis("off")
colunas = ["Algoritmo", "Familia", "Acoes", "Exploracao", "Efic. amostral"]
linhas = [
    ["Q-Learning / DQN", "Value-based", "Discretas", "epsilon-greedy", "Alta (off)"],
    ["REINFORCE",        "Policy-based", "Discretas", "Estocastica",   "Baixa (on)"],
    ["A2C",             "Actor-Critic",  "Discretas", "Estocastica",   "Baixa (on)"],
    ["PPO",             "Actor-Critic",  "Disc/Cont", "Estocastica",   "Media (on)"],
    ["DDPG",            "Policy-based",  "Continuas", "Ruido externo", "Alta (off)"],
    ["SAC",             "Policy-based",  "Continuas", "Entropia",      "Alta (off)"],
]
tabela = ax2.table(
    cellText=linhas,
    colLabels=colunas,
    cellLoc="center",
    loc="center",
    bbox=[0, 0, 1, 1]
)
tabela.auto_set_font_size(False)
tabela.set_fontsize(9)
for j in range(len(colunas)):
    tabela[0, j].set_facecolor("#2C3E50")
    tabela[0, j].set_text_props(color="white", fontweight="bold")
# linhas alternadas: azul muito claro e branco para facilitar leitura
cores_alt = ["#EBF5FB", "#FDFEFE"]
for i in range(len(linhas)):
    for j in range(len(colunas)):
        tabela[i+1, j].set_facecolor(cores_alt[i % 2])
ax2.set_title("Resumo dos algoritmos das Aulas 02 e 03", fontsize=11, pad=15)

plt.tight_layout()
plt.savefig("nb06_comparativo_final.png", dpi=100, bbox_inches="tight")
plt.show()
print("Gráfico salvo em nb06_comparativo_final.png")

In [6]:
# ── Resumo da progressão ─────────────────────────────────────────
print("Progressão da Aula 03 — CartPole-v1 (referência: resultados típicos)")
print()
print(f"{'Algoritmo':<30} {'Convergência típica'}")
print("-" * 55)
algoritmos = [
    ("Política aleatória",    f"média={np.mean(retornos_aleatorio):.0f} (medido)"),
    ("REINFORCE",             "~400-475 ep (variável)"),
    ("A2C com baseline",      "~300-400 ep (mais estável)"),
    ("PPO",                   "~200-300 ep (consistente)"),
    ("DDPG (Pendulum)",       "retorno ~-200 a -150"),
]
for nome, media in algoritmos:
    print(f"  {nome:<28} {media}")
print()
print("Nota: resultados variam com seed, arquitetura e hiperparâmetros.")

Progressão da Aula 03 — CartPole-v1 (referência: resultados típicos)

Algoritmo                      Convergência típica
-------------------------------------------------------
  Política aleatória           média=19 (medido)
  REINFORCE                    ~400-475 ep (variável)
  A2C com baseline             ~300-400 ep (mais estável)
  PPO                          ~200-300 ep (consistente)
  DDPG (Pendulum)              retorno ~-200 a -150

Nota: resultados variam com seed, arquitetura e hiperparâmetros.


## Bloco 4 — Interpretação pedagógica

### O que a tabela resume

A tabela de algoritmos organiza os métodos por três eixos práticos:

1. **Tipo de ação:** se as ações são discretas, Q-Learning e DQN bastam. Se são contínuas, é necessário DDPG ou SAC.
2. **Exploração:** *value-based* usa ε-greedy — uma regra externa. *Policy-based* usa a própria distribuição da política como mecanismo de exploração, ou ruído externo (DDPG).
3. **Eficiência amostral:** algoritmos *off-policy* (DQN, DDPG, SAC) reutilizam transições via *replay buffer*, tornando-os mais eficientes. Algoritmos *on-policy* (A2C, PPO) descartam dados após cada atualização — mais simples, mas menos eficientes.

### Por que a heurística funciona sem treino?

No CartPole, a regra "polo à direita → empurra à direita" captura intuitivamente o comportamento correto porque **o problema tem uma solução expressa em termos de uma única variável** (ângulo do polo). Mesmo assim, a heurística atinge apenas ~43/475 — longe do limiar de solução.

Em problemas reais, essa heurística não existe: um robô com 30 graus de liberdade, um sistema de recomendação com 10 milhões de usuários e 100 mil itens, um agente de trading com centenas de indicadores — não há regra simples que funcione. O RL aprende políticas em espaços que a intuição humana não consegue navegar.

### A mensagem-chave

> **Value-based e policy-based não se opõem — as melhores soluções modernas combinam os dois.**

O Actor-Critic já é uma combinação: crítico value-based orienta o ator policy-based. O SAC usa um crítico Q para treinar uma política estocástica. O AlphaZero combina política + função de valor + simulação do ambiente. A Aula 04 aprofunda essa convergência com model-based RL, RLHF e outros métodos avançados.

## Autoavaliação

<details>
<summary>Pergunta 1: Qual é a principal vantagem dos métodos policy-based sobre os value-based para espaços contínuos?</summary>

**Resposta:** Métodos policy-based (REINFORCE, PPO, DDPG, SAC) parametrizam diretamente π(a|s) ou μ(s) — a saída pode ser um vetor contínuo de qualquer dimensão. Métodos value-based exigem argmax sobre ações, que é trivial para ações discretas mas intratável para contínuos. Policy-based elimina esse requisito, tornando-os naturalmente adequados a robótica, controle de motores e outros domínios contínuos.

**Por quê:** A estrutura do policy gradient — maximizar E[A(s,a) · log π(a|s)] — não impõe nenhuma restrição sobre a dimensão ou tipo do espaço de ações.

</details>

<details>
<summary>Pergunta 2: Por que o PPO é considerado um bom equilíbrio entre performance e estabilidade?</summary>

**Resposta:** O PPO combina a eficiência de amostras do on-policy (múltiplas épocas por batch) com o controle de atualização do TRPO (via clipping), sem o custo computacional das restrições de segunda ordem. É mais estável que REINFORCE (menos variância) e mais simples que TRPO (sem hessiana). Em benchmarks discretos (CartPole, Atari) e contínuos (MuJoCo) o PPO frequentemente iguala ou supera alternativas mais complexas.

**Por quê:** Simplicidade de implementação + estabilidade + performance competitiva = algoritmo de referência para RL prático.

</details>

<details>
<summary>Pergunta 3: Quando seria preferível usar DDPG ao invés de SAC?</summary>

**Resposta:** DDPG pode ser preferível quando a política determinística é necessária (ex.: sistemas de controle onde a variância da ação é indesejável) ou quando o custo computacional do SAC (duas redes Q + ajuste automático de α) é proibitivo. Em ambientes com recompensa densa e pouca estocasticidade, o DDPG pode convergir mais rapidamente. Mas para a maioria dos benchmarks modernos, SAC é a escolha padrão.

**Por quê:** DDPG foi o ponto de partida histórico para controle contínuo com deep RL; SAC é sua evolução com melhoras de estabilidade e exploração. Preferir DDPG é geralmente uma questão de contexto específico ou restrições de custo.

</details>

## Mapeamento para o Desafio Final

A Aula 03 construiu um conjunto completo de competências em métodos policy-based. A tabela abaixo mapeia cada notebook ao que ele contribui para o Desafio Final.

| Notebook | Competência construída | Quando usar no Desafio Final |
|---|---|---|
| NB01 — Intuição de política | Entender política parametrizada e softmax | Qualquer tarefa com política estocástica |
| NB02 — REINFORCE / Policy Gradient | Implementar gradiente de política básico | Baseline de referência para comparação |
| NB03 — Baseline e A2C | Reduzir variância com função de vantagem | Sempre que REINFORCE for muito instável |
| NB04 — PPO | Limitar tamanho de atualizações | Ações discretas; qualquer problema sem controle contínuo |
| NB05 — DDPG e SAC | Política para espaços contínuos | Robótica, controle de motores, torques |

**Guia de decisão em 3 perguntas:**

1. **Ações discretas ou contínuas?**
   - Discretas → PPO (NB04) como ponto de partida
   - Contínuas → SAC (NB05) para exploração; DDPG quando a política determinística é preferível

2. **Dados limitados ou ambiente caro de simular?**
   - Priorizar eficiência amostral → SAC/DDPG (*off-policy* com replay buffer)
   - Ambiente barato de simular → PPO (*on-policy*) é mais simples

3. **Recompensa esparsa ou horizonte longo?**
   - Considerar métodos hierárquicos (Aula 04, NB05) ou curriculum learning antes de tunar hiperparâmetros

Os algoritmos desta aula são a base — a Aula 04 mostra como estendê-los para domínios mais complexos.

## Fio condutor do curso — MovieLens como MDP sequencial (Policy Gradient vs Q-Learning)

A Aula 02 aplicou Q-Learning ao MovieLens e obteve uma **política determinística** (argmax). A célula abaixo repete o experimento com uma **política estocástica (softmax)** treinada por gradiente de política — a mesma filosofia do REINFORCE.

| Aspecto | Q-Learning (Aula 02) | Policy Gradient (Aula 03) |
|---|---|---|
| Objeto aprendido | Q(estado, gênero) | θ[estado, gênero] — logits |
| Política resultante | Determinística (argmax) | Estocástica (softmax) |
| Exploração | ε-greedy explícito | Entropia da distribuição |
| Interpretação | Valor esperado por par (s, a) | Probabilidade de recomendar cada gênero |

Esta comparação é a síntese das Aulas 02 e 03 no mesmo problema de negócio.

In [ ]:
import pandas as pd

BASE = os.path.abspath("../..")
try:
    ratings = pd.read_csv(os.path.join(BASE, "data/movielens/ratings.csv"))
    movies  = pd.read_csv(os.path.join(BASE, "data/movielens/movies.csv"))
    DADOS_OK = True
except FileNotFoundError:
    print("[FALTA] Execute 01_contexto_movielens.ipynb para baixar os dados.")
    DADOS_OK = False

if DADOS_OK:
    GENEROS = ['Action', 'Comedy', 'Drama', 'Romance', 'Sci-Fi']
    N_A, N_S = len(GENEROS), 3

    def genero_dom(genres_str):
        for g in GENEROS:
            if g in str(genres_str):
                return GENEROS.index(g)
        return 0

    movies['acao'] = movies['genres'].apply(genero_dom)
    df = ratings.merge(movies[['movieId', 'acao']], on='movieId', how='left').dropna()
    df['gostou'] = (df['rating'] >= 4).astype(float)
    media_u = df.groupby('userId')['rating'].mean()
    df['estado'] = pd.cut(df['userId'].map(media_u), bins=3, labels=[0, 1, 2]).astype(int)
    df_sh = df.sample(frac=1, random_state=SEED)

    def softmax(x):
        # subtrai o máximo para evitar overflow numérico (softmax numericamente estável)
        e = np.exp(x - x.max())
        return e / e.sum()

    # ── Q-Learning (referência da Aula 02) — política determinística ────
    Q_ref = np.zeros((N_S, N_A))
    for _, row in df_sh.iterrows():
        s, a, r = int(row['estado']), int(row['acao']), float(row['gostou'])
        # regra de Bellman: Q(s,a) ← Q(s,a) + α·(r + γ·max Q(s') − Q(s,a))
        # α=0.05, γ=0.9; sem estado futuro explícito (episódico com 1 passo)
        Q_ref[s, a] += 0.05 * (r + 0.9 * Q_ref[s].max() - Q_ref[s, a])
    pi_ql = np.eye(N_A)[[np.argmax(Q_ref[s]) for s in range(N_S)]]

    # ── Policy Gradient (softmax) — política estocástica ────────────────
    THETA = np.zeros((N_S, N_A))
    baseline = np.zeros(N_S)
    for _, row in df_sh.iterrows():
        s, a, r = int(row['estado']), int(row['acao']), float(row['gostou'])
        pi = softmax(THETA[s])
        # gradiente do log-softmax: ∂log π(a|s) / ∂θ = e_a − π(s)
        grad = -pi.copy(); grad[a] += 1.0
        # α=0.02 menor que Q-Learning porque gradientes de política têm magnitude maior
        THETA[s] += 0.02 * (r - baseline[s]) * grad
        # baseline adaptativo: média móvel das recompensas por estado
        baseline[s] += 0.01 * (r - baseline[s])
    pi_pg = np.array([softmax(THETA[s]) for s in range(N_S)])

    # ── Visualização comparativa ─────────────────────────────────────────
    nomes_s = ['Crítico', 'Mediano', 'Entusiasta']
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, titulo, pi, label_cb in zip(
        axes,
        ['Q-Learning (Aula 02) — política determinística',
         'Policy Gradient (Aula 03) — política estocástica'],
        [pi_ql, pi_pg],
        ['P(ação|greedy)', 'π(ação|estado)'],
    ):
        im = ax.imshow(pi, cmap='Blues', aspect='auto', vmin=0, vmax=1)
        ax.set_xticks(range(N_A)); ax.set_xticklabels(GENEROS, rotation=15)
        ax.set_yticks(range(N_S)); ax.set_yticklabels(nomes_s)
        for i in range(N_S):
            for j in range(N_A):
                ax.text(j, i, f"{pi[i, j]:.2f}", ha='center', va='center', fontsize=9,
                        color='white' if pi[i, j] > 0.5 else 'black')
        ax.set_title(titulo, fontsize=11)
        plt.colorbar(im, ax=ax, label=label_cb)

    plt.suptitle('MovieLens — value-based (determinístico) vs policy-based (estocástico)', fontsize=12)
    plt.tight_layout()
    plt.show()

    print("Resumo das políticas aprendidas:")
    for i, nome in enumerate(nomes_s):
        rec_ql = GENEROS[np.argmax(pi_ql[i])]
        rec_pg = GENEROS[np.argmax(pi_pg[i])]
        entropia = -np.sum(pi_pg[i] * np.log(pi_pg[i] + 1e-9))
        print(f"  {nome:<14} Q-Learning → {rec_ql:<10} | Policy Gradient → {rec_pg:<10} (entropia={entropia:.3f})")

### Interpretação do heatmap — determinístico vs estocástico na recomendação

**Q-Learning (esquerda):** para cada perfil de usuário, há sempre um gênero com valor 1,0 e os demais com 0,0. A política é **determinística** — o sistema sempre recomenda o mesmo gênero para o mesmo tipo de usuário. Vantagem: previsível e auditável. Desvantagem: sem diversidade, tende ao *filter bubble* (usuário nunca descobre novos gêneros).

**Policy Gradient (direita):** probabilidades distribuídas entre múltiplos gêneros (ex: 0,45 para um gênero, 0,30 para outro). A política é **estocástica** — o sistema varia as recomendações mesmo para o mesmo perfil. Vantagem: diversidade natural, *serendipidade*. Desvantagem: menos previsível, mais difícil de auditar.

**O trade-off na prática:** sistemas de recomendação reais enfrentam exatamente este dilema. O YouTube (2019) migrou para políticas estocásticas baseadas em Policy Gradient precisamente para escapar do *filter bubble* — usuários que recebem recomendações diversificadas têm maior engajamento no longo prazo, mesmo que cada recomendação individual seja menos precisa.

A coluna `entropia` no resumo abaixo quantifica essa diversidade: entropia alta = política mais exploratória, entropia zero = política completamente determinística.

## Bloco 5 — Limites desta aula e transição para a Aula 04

### O que ficou fora da Aula 03

- **Model-based RL:** os algoritmos desta aula aprendem por tentativa e erro — sem modelo do ambiente. Algoritmos como Dyna, MuZero e World Models usam um modelo aprendido para planejar antes de agir.
- **Offline RL:** todos os algoritmos aqui aprendem *on-line* (interagindo com o ambiente). RL *offline* aprende de conjuntos de dados fixos, sem nova interação — crucial em robótica real.
- **RLHF:** *Reinforcement Learning from Human Feedback* — a técnica por trás do alinhamento de modelos de linguagem (ChatGPT, Claude). A recompensa vem de preferências humanas, não de um ambiente simulado.

### O que a Aula 04 cobre

A Aula 04 aborda:
- **RL baseado em modelos** (*model-based RL*): aprender um modelo do ambiente e usá-lo para planejamento.
- **RL com recompensa aprendida** (RLHF, IRL): quando a função de recompensa não está disponível diretamente.
- **Aprendizado por transferência em RL:** como reaproveitar políticas entre tarefas.

> "Os métodos desta aula são os tijolos. A Aula 04 mostra como construir edifícios." 

In [8]:
# Glossário dos termos técnicos deste notebook
rl_utils.exibir_glossario([
    'policy', 'policy gradient', 'actor-critic',
    'PPO', 'DDPG', 'SAC', 'replay buffer', 'value function',
])

Termo (EN)        Tradução (PT)                Descrição
---------------------------------------------------------------------------------------------------------
DDPG              DDPG                         Deep Deterministic Policy Gradient — política determinística para espaços contínuos.
PPO               PPO                          Proximal Policy Optimization — limita a atualização da política com clipping.
SAC               SAC                          Soft Actor-Critic — maximiza recompensa e entropia da política.
actor-critic      ator-crítico                 Arquitetura com rede de política (ator) e de valor (crítico) simultâneas.
policy            política                     π(a|s) — distribuição de probabilidade sobre ações dado o estado.
policy gradient   gradiente de política        Família de métodos que otimiza diretamente os parâmetros da política.
replay buffer     buffer de replay             Memória que armazena transições para treinamento em mini-batches.
value

## Leituras e referências

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2ª ed.). MIT Press. Cap. 13. Disponível em: http://incompleteideas.net/book/the-book-2nd.html. Acesso em: abril 2026.
- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). Proximal Policy Optimization Algorithms. *arXiv:1707.06347*. Disponível em: https://arxiv.org/abs/1707.06347. Acesso em: abril 2026.
- Ouyang, L., et al. (2022). Training language models to follow instructions with human feedback. *arXiv:2203.02155*. Disponível em: https://arxiv.org/abs/2203.02155. Acesso em: abril 2026.
- Farama Foundation. *Gymnasium documentation*. Disponível em: https://gymnasium.farama.org. Acesso em: abril 2026.